# 🌊 FloodHunger Ghana
## Flood-Driven Food Insecurity Early Warning System
### District-Level ML Pipeline | 53 Districts × 22 Years

---
| | |
|---|---|
| **Datasets** | CHIRPS v3 Rainfall · WFP VAM Food Prices · ACLED Conflict Events |
| **Scope** | 53 districts · 16 regions · 2003–2024 · ~14,000 rows |
| **Models** | XGBoost IPC Classifier · Random Forest Price Regressor · Isolation Forest Anomaly Detector |
| **Output** | WFP district alert feed · Farmer voice bulletins · SHAP explainability |


## 1 · Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, mean_absolute_error, r2_score
)
import xgboost as xgb
import shap

# ── Paths ──────────────────────────────────────────────────
RAW  = Path("data/raw")
PROC = Path("data/processed")
MDL  = Path("data/models")
OUT  = Path("data/outputs")
for p in [PROC, MDL, OUT]:
    p.mkdir(parents=True, exist_ok=True)

# ── Style ──────────────────────────────────────────────────
BLUE    = "#1F5C8B"
LBLUE   = "#85C1E9"
DBLUE   = "#0D3C6E"
PALETTE = [BLUE, "#2E86C1", LBLUE, "#D6E8F5"]

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.family": "sans-serif",
})

SEED = 42
np.random.seed(SEED)

IPC_LABELS  = {1: "Minimal", 2: "Stressed", 3: "Crisis", 4: "Emergency"}
IPC_ACTIONS = {
    1: "Monitor",
    2: "Alert field teams — pre-position supplies",
    3: "Deploy food stocks and vouchers",
    4: "EMERGENCY — cash transfers NOW",
}
IPC_COLORS = {1: "🟢", 2: "🟡", 3: "🟠", 4: "🔴"}

print("✓ Setup complete")
print(f"  RAW      : {RAW}")
print(f"  PROCESSED: {PROC}")
print(f"  MODELS   : {MDL}")
print(f"  OUTPUTS  : {OUT}")

## 2 · Load & Inspect Raw Data

In [ ]:
chirps_raw = pd.read_csv(RAW / "chirps_ghana_rainfall.csv")
prices_raw = pd.read_csv(RAW / "wfp_food_prices_ghana.csv")
acled_raw  = pd.read_csv(RAW / "acled_ghana_events.csv")

print("=" * 60)
print("  RAW DATASET OVERVIEW")
print("=" * 60)
print(f"  CHIRPS Rainfall : {len(chirps_raw):>7,} rows")
print(f"    Districts     : {chirps_raw['adm2_name'].nunique()}")
print(f"    Regions       : {chirps_raw['adm1_name'].nunique()}")
print()
print(f"  WFP Food Prices : {len(prices_raw):>7,} rows")
print(f"    Commodities   : {prices_raw['commodity'].unique().tolist()}")
print(f"    Markets       : {prices_raw['market'].unique().tolist()}")
print()
print(f"  ACLED Conflicts : {len(acled_raw):>7,} rows")
print(f"    Event types   : {acled_raw['event_type'].unique().tolist()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# CHIRPS rainfall distribution
axes[0].hist(
    pd.to_numeric(chirps_raw["rfh"], errors="coerce").dropna(),
    bins=60, color=BLUE, alpha=0.85, edgecolor="white"
)
axes[0].set_title("CHIRPS — Rainfall (mm/dekad)")
axes[0].set_xlabel("Rainfall mm")
axes[0].set_ylabel("Count")

# WFP prices by commodity
prices_raw["price"] = pd.to_numeric(prices_raw["price"], errors="coerce")
pm = prices_raw.groupby("commodity")["price"].mean().sort_values()
axes[1].barh(pm.index, pm.values, color=BLUE, alpha=0.85)
axes[1].set_title("WFP — Avg Price by Commodity (GHS/kg)")
axes[1].set_xlabel("GHS")

# ACLED event types
ec = acled_raw["event_type"].value_counts()
axes[2].barh(ec.index, ec.values, color=BLUE, alpha=0.85)
axes[2].set_title("ACLED — Events by Type")
axes[2].set_xlabel("Event Count")

plt.suptitle("FloodHunger Ghana — Raw Data Overview", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUT / "00_raw_data_overview.png", bbox_inches="tight")
plt.show()
print("✓ Saved → data/outputs/00_raw_data_overview.png")

## 3 · Clean & Merge at District Level

**Strategy:**
- **CHIRPS** → spine at district level (53 districts × 264 months = ~13,992 rows)
- **WFP Prices** → region-level, broadcast to every district within that region
- **ACLED** → region-level, broadcast similarly


In [ ]:
# ── Region name standardiser ───────────────────────────────
REGION_MAP = {
    "Greater Accra": "Greater Accra",  "Ashanti": "Ashanti",
    "Northern": "Northern",            "Upper East": "Upper East",
    "Upper West": "Upper West",        "Volta": "Volta",
    "Eastern": "Eastern",             "Central": "Central",
    "Western": "Western",             "Brong-Ahafo": "Bono",
    "Brong Ahafo": "Bono",            "Bono": "Bono",
    "Bono East": "Bono East",         "Ahafo": "Ahafo",
    "Savannah": "Savannah",           "Oti": "Oti",
    "North East": "North East",       "Western North": "Western North",
    "Accra": "Greater Accra",         "Kumasi": "Ashanti",
    "Tamale": "Northern",             "Bolgatanga": "Upper East",
    "Wa": "Upper West",               "Ho": "Volta",
    "Koforidua": "Eastern",           "Cape Coast": "Central",
    "Sekondi-Takoradi": "Western",    "Sunyani": "Bono",
}

def std_region(name):
    if pd.isna(name): return "Unknown"
    return REGION_MAP.get(str(name).strip(), str(name).strip())

print("✓ Region standardiser ready")

In [ ]:
# ── CHIRPS: dekadal → monthly per district ─────────────────
chirps = chirps_raw.copy()
chirps["date"]     = pd.to_datetime(chirps["date"], errors="coerce")
chirps             = chirps.dropna(subset=["date"])
chirps["year"]     = chirps["date"].dt.year
chirps["month"]    = chirps["date"].dt.month
chirps["region"]   = chirps["adm1_name"].apply(std_region)
chirps["district"] = chirps["adm2_name"].str.strip()

rain_cols = ["rfh","rfh_avg","rfq","r1h","r1h_avg","r1q","r3h","r3h_avg","r3q"]
for c in rain_cols:
    chirps[c] = pd.to_numeric(chirps[c], errors="coerce")

chirps_m = (
    chirps.groupby(["region","district","year","month"])[rain_cols]
    .mean().round(3).reset_index()
)
chirps_m = chirps_m.rename(columns={
    "rfh":  "rainfall_mm",       "rfh_avg": "rainfall_avg_mm",
    "rfq":  "rainfall_anomaly_pct",
    "r1h":  "rainfall_1m_mm",    "r1h_avg": "rainfall_1m_avg_mm",
    "r1q":  "rainfall_1m_anomaly_pct",
    "r3h":  "rainfall_3m_mm",    "r3h_avg": "rainfall_3m_avg_mm",
    "r3q":  "rainfall_3m_anomaly_pct",
})
chirps_m["flood_flag"]   = (chirps_m["rainfall_anomaly_pct"] > 50).astype(int)
chirps_m["drought_flag"] = (chirps_m["rainfall_anomaly_pct"] < -40).astype(int)

chirps_m.to_csv(PROC / "clean_chirps.csv", index=False)
print(f"✓ CHIRPS clean")
print(f"  Rows      : {len(chirps_m):,}")
print(f"  Districts : {chirps_m['district'].nunique()}")
print(f"  Regions   : {chirps_m['region'].nunique()}")
print(f"  Years     : {chirps_m['year'].min()} – {chirps_m['year'].max()}")
chirps_m.head(3)

In [ ]:
# ── WFP Prices: region-level monthly pivot ─────────────────
prices = prices_raw.copy()
prices["date"]   = pd.to_datetime(prices["date"], errors="coerce")
prices           = prices.dropna(subset=["date"])
prices["year"]   = prices["date"].dt.year
prices["month"]  = prices["date"].dt.month
prices["region"] = prices["admin1"].apply(std_region)
prices["price"]  = pd.to_numeric(prices["price"], errors="coerce")
prices           = prices[prices["price"] > 0].dropna(subset=["price"])

key = ["Maize","Millet","Sorghum","Rice","Cowpeas","Groundnuts"]
prices = prices[prices["commodity"].apply(
    lambda x: any(k.lower() in x.lower() for k in key)
)]

price_m = (
    prices.groupby(["region","year","month","commodity"])["price"]
    .mean().round(4).reset_index()
)
pivot = price_m.pivot_table(
    index=["region","year","month"], columns="commodity",
    values="price", aggfunc="mean"
).reset_index()
pivot.columns.name = None

rename = {}
for c in pivot.columns:
    cl = str(c).lower()
    if "maize" in cl:       rename[c] = "price_maize"
    elif "millet" in cl:    rename[c] = "price_millet"
    elif "sorghum" in cl:   rename[c] = "price_sorghum"
    elif "rice" in cl:      rename[c] = "price_rice"
    elif "cowpea" in cl:    rename[c] = "price_cowpeas"
    elif "groundnut" in cl: rename[c] = "price_groundnuts"
pivot = pivot.rename(columns=rename)
pivot = pivot.sort_values(["region","year","month"]).reset_index(drop=True)

price_col = next((c for c in ["price_maize","price_millet","price_sorghum"]
                  if c in pivot.columns and pivot[c].notna().sum() > 50), None)
if price_col:
    pivot["price_lag1"]          = pivot.groupby("region")[price_col].shift(1)
    pivot["price_change_pct"]    = ((pivot[price_col] - pivot["price_lag1"]) / pivot["price_lag1"] * 100).round(2)
    pivot["price_volatility_3m"] = (
        pivot.groupby("region")["price_change_pct"]
        .transform(lambda x: x.rolling(3, min_periods=1).std()).round(3)
    )
    pivot["price_shock_flag"] = (pivot["price_change_pct"] > 20).astype(int)
    print(f"  Price change computed from: {price_col}")

pivot.to_csv(PROC / "clean_prices.csv", index=False)
print(f"✓ WFP Prices clean: {len(pivot):,} rows  |  {pivot['region'].nunique()} regions")
pivot.head(3)

In [ ]:
# ── ACLED: region-level monthly conflict counts ────────────
acled = acled_raw.copy()
acled["date"]        = pd.to_datetime(acled["event_date"], errors="coerce")
acled                = acled.dropna(subset=["date"])
acled["year"]        = acled["date"].dt.year
acled["month"]       = acled["date"].dt.month
acled["region"]      = acled["admin1"].apply(std_region)
acled["fatalities"]  = pd.to_numeric(acled["fatalities"], errors="coerce").fillna(0)
acled["is_flood_event"] = acled["event_type"].str.lower().str.contains(
    "flood|displacement|displaced", na=False).astype(int)

acled_m = (
    acled.groupby(["region","year","month"])
    .agg(
        conflict_events       = ("event_id_cnty","count"),
        fatalities_total      = ("fatalities","sum"),
        flood_displace_events = ("is_flood_event","sum"),
    ).reset_index()
)
acled_m = acled_m.sort_values(["region","year","month"]).reset_index(drop=True)
acled_m["conflict_events_lag1"] = acled_m.groupby("region")["conflict_events"].shift(1).fillna(0)
acled_m["flood_displace_lag1"]  = acled_m.groupby("region")["flood_displace_events"].shift(1).fillna(0)
q75 = acled_m["conflict_events"].quantile(0.75)
acled_m["high_conflict_flag"] = (acled_m["conflict_events"] >= q75).astype(int)

acled_m.to_csv(PROC / "clean_acled.csv", index=False)
print(f"✓ ACLED clean: {len(acled_m):,} rows  |  Total events: {acled_m['conflict_events'].sum():,}")
acled_m.head(3)

In [ ]:
# ── MERGE: district spine + region-level broadcasts ────────
master = chirps_m.copy()
master = master.merge(pivot,   on=["region","year","month"], how="left")
master = master.merge(acled_m, on=["region","year","month"], how="left")

for c in ["conflict_events","fatalities_total","flood_displace_events",
          "conflict_events_lag1","flood_displace_lag1","high_conflict_flag"]:
    if c in master.columns:
        master[c] = master[c].fillna(0)

# ── Season phase ───────────────────────────────────────────
NORTH   = {"Northern","Upper East","Upper West","Savannah","North East"}

def get_season(row):
    m, r = row["month"], row["region"]
    if r in NORTH:
        if m in [3,4,5]:    return "pre_planting"
        elif m in [6,7]:    return "planting"
        elif m in [8,9]:    return "growing"
        elif m in [10,11]:  return "harvest"
        else:               return "lean"
    else:
        if m in [3,4]:      return "planting_s1"
        elif m in [5,6]:    return "growing_s1"
        elif m == 7:        return "harvest_s1"
        elif m == 8:        return "inter_season"
        elif m in [9,10]:   return "growing_s2"
        elif m == 11:       return "harvest_s2"
        else:               return "lean"

master["season_phase"]   = master.apply(get_season, axis=1)
master["is_lean_season"] = master["season_phase"].isin(["lean","pre_planting"]).astype(int)

# ── District vulnerability ─────────────────────────────────
VULN = {"Northern","Upper East","Upper West","Savannah","North East","Oti"}
master["district_vulnerability"] = master["region"].apply(lambda r: 2 if r in VULN else 1)

# ── IPC Phase (FEWS NET rule-based simulation) ─────────────
def compute_ipc(row):
    score = 0
    rain  = row.get("rainfall_anomaly_pct", 0) or 0
    if rain < -40:    score += 2
    elif rain < -20:  score += 1
    elif rain > 60:   score += 1
    score += row.get("flood_flag", 0) * 2
    score += row.get("drought_flag", 0) * 1
    pc = row.get("price_change_pct", 0) or 0
    if pc > 30:       score += 3
    elif pc > 15:     score += 2
    elif pc > 5:      score += 1
    score += min(row.get("conflict_events", 0) or 0, 10) * 0.3
    score += row.get("is_lean_season", 0)
    score += row.get("district_vulnerability", 1) - 1
    if score <= 1:    return 1
    elif score <= 3:  return 2
    elif score <= 6:  return 3
    else:             return 4

master["ipc_phase"]             = master.apply(compute_ipc, axis=1)
master["flood_price_interaction"] = (master["flood_flag"] * master.get("price_change_pct", pd.Series(0, index=master.index)).fillna(0)).round(3)

master = master.sort_values(["district","year","month"]).reset_index(drop=True)
master = master.dropna(subset=["rainfall_mm","region","district","year","month"])
master.to_csv(PROC / "master_monthly.csv", index=False)

print("=" * 55)
print("  DISTRICT MASTER TABLE")
print("=" * 55)
print(f"  Total rows  : {len(master):,}")
print(f"  Districts   : {master['district'].nunique()}")
print(f"  Regions     : {master['region'].nunique()}")
print(f"  Columns     : {len(master.columns)}")
print(f"  Years       : {master['year'].min()} – {master['year'].max()}")
print()
print("  IPC Phase distribution:")
for ph, cnt in master["ipc_phase"].value_counts().sort_index().items():
    pct = cnt / len(master) * 100
    bar = "█" * int(pct / 2)
    print(f"    {IPC_COLORS[ph]} Phase {ph} — {IPC_LABELS[ph]:<12}: {cnt:>6,} ({pct:4.1f}%)  {bar}")

master.head(3)

## 4 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))

# IPC distribution
phase_counts = master["ipc_phase"].value_counts().sort_index()
colors_ph = ["#2ECC71","#F4D03F","#E67E22","#E74C3C"]
bars = axes[0,0].bar(
    [f"P{p}\n{IPC_LABELS[p]}" for p in phase_counts.index],
    phase_counts.values, color=colors_ph, alpha=0.9, edgecolor="white"
)
for bar, val in zip(bars, phase_counts.values):
    axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                   f"{val:,}", ha="center", fontsize=9, fontweight="bold")
axes[0,0].set_title("IPC Phase Distribution — All Districts")
axes[0,0].set_ylabel("District-Month Count")

# Rainfall anomaly by year
rain_yr = master.groupby("year")["rainfall_anomaly_pct"].mean()
axes[0,1].bar(rain_yr.index, rain_yr.values,
              color=[BLUE if v >= 0 else "#E74C3C" for v in rain_yr.values], alpha=0.85)
axes[0,1].axhline(0, color="black", lw=0.8, linestyle="--")
axes[0,1].set_title("Average Rainfall Anomaly by Year")
axes[0,1].set_xlabel("Year"); axes[0,1].set_ylabel("Anomaly (%)")

# Flood months by region
flood_reg = master.groupby("region")["flood_flag"].sum().sort_values(ascending=True)
axes[0,2].barh(flood_reg.index, flood_reg.values, color=BLUE, alpha=0.85)
axes[0,2].set_title("Total Flood Months by Region")
axes[0,2].set_xlabel("Count")

# IPC by region
ipc_reg = master.groupby("region")["ipc_phase"].mean().sort_values(ascending=False)
c_reg   = ["#E74C3C" if v >= 3 else "#E67E22" if v >= 2.5 else BLUE for v in ipc_reg.values]
axes[1,0].barh(ipc_reg.index, ipc_reg.values, color=c_reg, alpha=0.85)
axes[1,0].axvline(2, color="orange", lw=1.5, linestyle="--", label="Stressed")
axes[1,0].axvline(3, color="red",    lw=1.5, linestyle="--", label="Crisis")
axes[1,0].set_title("Average IPC Phase by Region"); axes[1,0].legend(fontsize=8)

# Price change distribution
if "price_change_pct" in master.columns:
    pc = master["price_change_pct"].dropna()
    pc = pc[pc.abs() <= 100]
    axes[1,1].hist(pc, bins=60, color=BLUE, alpha=0.85, edgecolor="white")
    axes[1,1].axvline(0,  color="black", lw=1, linestyle="--")
    axes[1,1].axvline(20, color="red",   lw=1.5, linestyle="--", label="Shock (20%)")
    axes[1,1].set_title("Food Price Change Distribution (%)")
    axes[1,1].set_xlabel("Monthly Change (%)"); axes[1,1].legend(fontsize=9)

# Conflict over time
conf_yr = master.groupby("year")["conflict_events"].sum()
axes[1,2].plot(conf_yr.index, conf_yr.values, color=BLUE, lw=2.5, marker="o", markersize=5)
axes[1,2].fill_between(conf_yr.index, conf_yr.values, alpha=0.12, color=BLUE)
axes[1,2].set_title("Total Conflict Events per Year"); axes[1,2].set_xlabel("Year")

plt.suptitle("FloodHunger Ghana — EDA Overview", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(OUT / "01_eda_overview.png", bbox_inches="tight")
plt.show()
print("✓ Saved → data/outputs/01_eda_overview.png")

In [ ]:
# Seasonal patterns
mn = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ipc_m = master.groupby("month")["ipc_phase"].mean()
axes[0].plot(ipc_m.index, ipc_m.values, color=BLUE, lw=2.5, marker="o", ms=7)
axes[0].fill_between(ipc_m.index, ipc_m.values, 1, alpha=0.1, color=BLUE)
axes[0].axhline(2, color="orange", lw=1.5, ls="--", alpha=0.7, label="Stressed")
axes[0].axhline(3, color="red",    lw=1.5, ls="--", alpha=0.7, label="Crisis")
axes[0].set_xticks(range(1,13)); axes[0].set_xticklabels(mn)
axes[0].set_ylabel("Average IPC Phase"); axes[0].set_title("Seasonal IPC Pattern — All Ghana")
axes[0].legend()

flood_m = master.groupby("month")["flood_flag"].mean() * 100
axes[1].bar(range(1,13), flood_m.values, color=BLUE, alpha=0.85, edgecolor="white")
axes[1].set_xticks(range(1,13)); axes[1].set_xticklabels(mn)
axes[1].set_ylabel("% Districts with Flood Flag")
axes[1].set_title("Flood Occurrence by Month — All Districts")

plt.suptitle("Seasonal Patterns — FloodHunger Ghana", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT / "02_seasonal_patterns.png", bbox_inches="tight")
plt.show()

## 5 · Feature Engineering

In [ ]:
df = master.copy()
df = df.sort_values(["district","year","month"]).reset_index(drop=True)

# Rainfall lags
for lag in [1, 2, 3]:
    df[f"rainfall_anomaly_lag{lag}"] = df.groupby("district")["rainfall_anomaly_pct"].shift(lag)
    df[f"flood_flag_lag{lag}"]       = df.groupby("district")["flood_flag"].shift(lag).fillna(0)
    df[f"drought_flag_lag{lag}"]     = df.groupby("district")["drought_flag"].shift(lag).fillna(0)

df["rainfall_rolling_3m"] = df.groupby("district")["rainfall_mm"].transform(
    lambda x: x.rolling(3, min_periods=1).mean()).round(3)
df["rainfall_rolling_6m"] = df.groupby("district")["rainfall_mm"].transform(
    lambda x: x.rolling(6, min_periods=1).mean()).round(3)

def flood_streak(s):
    streak, count = [], 0
    for v in s:
        count = count + 1 if v == 1 else 0
        streak.append(count)
    return streak

df["flood_streak_months"] = df.groupby("district")["flood_flag"].transform(flood_streak)

# Price lags
if "price_change_pct" in df.columns:
    for lag in [1, 2]:
        df[f"price_change_lag{lag}"]     = df.groupby("district")["price_change_pct"].shift(lag)
        df[f"price_volatility_lag{lag}"] = df.groupby("district")["price_volatility_3m"].shift(lag)
    df["flood_price_lag1_interact"] = (df["flood_flag_lag1"] * df["price_change_pct"].fillna(0)).round(3)
    df["drought_price_interact"]    = (df["drought_flag"]    * df["price_change_pct"].fillna(0)).round(3)

# Conflict lags
if "conflict_events" in df.columns:
    for lag in [1, 2]:
        df[f"conflict_lag{lag}"] = df.groupby("district")["conflict_events"].shift(lag).fillna(0)
    df["conflict_rolling_3m"] = df.groupby("district")["conflict_events"].transform(
        lambda x: x.rolling(3, min_periods=1).sum())

# Season encoding
season_dummies = pd.get_dummies(df["season_phase"], prefix="season")
df = pd.concat([df, season_dummies], axis=1)
df["month_sin"]  = np.sin(2 * np.pi * df["month"] / 12).round(4)
df["month_cos"]  = np.cos(2 * np.pi * df["month"] / 12).round(4)
df["time_trend"] = (df["year"] - df["year"].min()) * 12 + df["month"]

# Region profiles
flood_freq = df.groupby("region")["flood_flag"].mean().rename("region_flood_freq_hist")
df = df.merge(flood_freq, on="region", how="left")
if "conflict_events" in df.columns:
    ci = df.groupby("region")["conflict_events"].mean().rename("region_conflict_intensity_hist")
    df = df.merge(ci, on="region", how="left")

NORTH2   = {"Northern","Upper East","Upper West","Savannah","North East"}
COASTAL2 = {"Greater Accra","Central","Western"}
df["region_type"] = df["region"].apply(lambda r: 0 if r in NORTH2 else (2 if r in COASTAL2 else 1))

# Compound risk score
df["compound_risk_score"] = (
    df["rainfall_anomaly_pct"].clip(-100,100).abs() * 0.01 +
    df["flood_flag"] * 2.0 +
    df.get("price_change_pct", pd.Series(0, index=df.index)).fillna(0).clip(0,100) * 0.05 +
    df.get("conflict_events", pd.Series(0, index=df.index)).fillna(0) * 0.1 +
    df["is_lean_season"] * 1.0
).round(3)

lag_cols = [c for c in df.columns if "lag" in c or "rolling" in c]
df[lag_cols] = df[lag_cols].fillna(0)
df = df.drop(columns=["season_phase"], errors="ignore")
df.to_csv(PROC / "features.csv", index=False)

exclude      = ["region","district","year","month","ipc_phase","compound_risk_flag"]
feature_cols = [c for c in df.columns
                if c not in exclude
                and df[c].dtype in [np.float64, np.int64, float, int]
                and df[c].nunique() > 1]

print(f"✓ Feature engineering complete")
print(f"  Rows     : {len(df):,}")
print(f"  Districts: {df['district'].nunique()}")
print(f"  Features : {len(feature_cols)}")
print(f"  Saved    → data/processed/features.csv")

## 6 · Model Training

### 6.1 XGBoost IPC Phase Classifier

In [ ]:
existing_phases = sorted(df["ipc_phase"].unique())
n_classes       = len(existing_phases)
phase_to_idx    = {p: i for i, p in enumerate(existing_phases)}
idx_to_phase    = {i: p for p, i in phase_to_idx.items()}

X = df[feature_cols].fillna(0)
y = df["ipc_phase"].map(phase_to_idx)

phase_counts   = df["ipc_phase"].value_counts()
total          = len(df)
class_weights  = {phase_to_idx[p]: total / (n_classes * cnt) for p, cnt in phase_counts.items()}
sample_weights = df["ipc_phase"].map({p: class_weights[phase_to_idx[p]] for p in existing_phases})

X_train, X_test, y_train, y_test, sw_train, _ = train_test_split(
    X, y, sample_weights, test_size=0.2, random_state=SEED, stratify=y
)

print(f"  IPC phases  : {existing_phases}")
print(f"  Train rows  : {len(X_train):,}")
print(f"  Test rows   : {len(X_test):,}")
print(f"  Class weights: {class_weights}")

In [ ]:
objective   = "binary:logistic" if n_classes == 2 else "multi:softmax"
eval_metric = "logloss"         if n_classes == 2 else "mlogloss"

ipc_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    gamma=1, reg_alpha=0.1, reg_lambda=1,
    objective=objective,
    num_class=n_classes if n_classes > 2 else None,
    random_state=SEED, eval_metric=eval_metric, verbosity=0,
)
ipc_model.fit(X_train, y_train, sample_weight=sw_train,
              eval_set=[(X_test, y_test)], verbose=False)

y_pred       = ipc_model.predict(X_test)
y_pred_phase = pd.Series(y_pred).map(idx_to_phase).values
y_test_phase = pd.Series(y_test.values).map(idx_to_phase).values
f1           = f1_score(y_test, y_pred, average="weighted")

print(f"✓ XGBoost trained  |  Weighted F1: {f1:.4f}")
print()
print(classification_report(y_test_phase, y_pred_phase,
      target_names=[f"Phase {p}" for p in existing_phases], zero_division=0))

joblib.dump(ipc_model,    MDL / "ipc_classifier_xgb.pkl")
joblib.dump(feature_cols, MDL / "feature_cols.pkl")
joblib.dump(phase_to_idx, MDL / "phase_to_idx.pkl")
print("✓ Saved → data/models/ipc_classifier_xgb.pkl")

In [ ]:
cm = confusion_matrix(y_test_phase, y_pred_phase, labels=existing_phases)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=[f"P{p}" for p in existing_phases],
            yticklabels=[f"P{p}" for p in existing_phases])
ax.set_title(f"IPC Classifier — Confusion Matrix  (Weighted F1 = {f1:.4f})")
ax.set_xlabel("Predicted Phase"); ax.set_ylabel("True Phase")
plt.tight_layout()
plt.savefig(OUT / "model1_confusion_matrix.png", bbox_inches="tight")
plt.show()

### 6.2 Random Forest Price Regressor

In [ ]:
candidate_targets = ["price_change_pct","price_millet","price_sorghum","price_cowpeas"]
price_target = next((c for c in candidate_targets
                     if c in df.columns and df[c].notna().sum() > 100), None)

df_reg = df.copy()
if price_target in ["price_millet","price_sorghum","price_cowpeas"]:
    df_reg["target_price_change"] = df_reg.groupby("district")[price_target].transform(
        lambda x: x.pct_change() * 100)
else:
    df_reg["target_price_change"] = df_reg.groupby("district")[price_target].shift(-1)

df_reg = df_reg.dropna(subset=["target_price_change"])
df_reg = df_reg[df_reg["target_price_change"].abs() <= 200]

reg_features = [c for c in feature_cols if c != price_target and df_reg[c].notna().mean() > 0.5]
Xr = df_reg[reg_features].fillna(0)
yr = df_reg["target_price_change"]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=SEED)

price_model = RandomForestRegressor(
    n_estimators=200, max_depth=10, min_samples_leaf=5,
    max_features="sqrt", random_state=SEED, n_jobs=-1
)
price_model.fit(Xr_train, yr_train)
yr_pred = price_model.predict(Xr_test)
mae = mean_absolute_error(yr_test, yr_pred)
r2  = r2_score(yr_test, yr_pred)

print(f"✓ Price Regressor trained  (target: {price_target})")
print(f"  MAE   : {mae:.3f}%")
print(f"  R²    : {r2:.4f}")
print(f"  Train : {len(Xr_train):,}  |  Test: {len(Xr_test):,}")

joblib.dump(price_model,  MDL / "price_regressor_rf.pkl")
joblib.dump(reg_features, MDL / "reg_feature_cols.pkl")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(yr_test[:500], yr_pred[:500], alpha=0.35, color=BLUE, s=18)
lims = [float(min(yr_test.min(), yr_pred.min())), float(max(yr_test.max(), yr_pred.max()))]
axes[0].plot(lims, lims, "r--", lw=1.5, alpha=0.8, label="Perfect prediction")
axes[0].set_xlabel("Actual Price Change (%)"); axes[0].set_ylabel("Predicted (%)")
axes[0].set_title(f"Price Regressor — Actual vs Predicted\nMAE={mae:.2f}%   R²={r2:.3f}")
axes[0].legend()

imp   = price_model.feature_importances_
top_i = np.argsort(imp)[-15:][::-1]
axes[1].barh(range(15), imp[top_i][::-1], color=BLUE, alpha=0.85)
axes[1].set_yticks(range(15))
axes[1].set_yticklabels([reg_features[i] for i in top_i][::-1], fontsize=9)
axes[1].set_xlabel("Feature Importance (Gini)")
axes[1].set_title("Price Regressor — Top 15 Features")

plt.tight_layout()
plt.savefig(OUT / "model2_price_regressor.png", bbox_inches="tight")
plt.show()

### 6.3 Isolation Forest Anomaly Detector

In [ ]:
anomaly_feats = [c for c in feature_cols if any(kw in c for kw in
    ["rainfall","flood","price","conflict","risk","season","drought"])]

Xa     = df[anomaly_feats].fillna(0)
scaler = StandardScaler()
Xa_sc  = scaler.fit_transform(Xa)

anomaly_model = IsolationForest(
    n_estimators=200, contamination=0.03,
    max_features=0.8, random_state=SEED, n_jobs=-1
)
anomaly_model.fit(Xa_sc)

df["anomaly_label"] = anomaly_model.predict(Xa_sc)
df["anomaly_score"] = (-anomaly_model.score_samples(Xa_sc)).round(4)
df["is_anomaly"]    = (df["anomaly_label"] == -1).astype(int)
n_anom = df["is_anomaly"].sum()

print(f"✓ Isolation Forest trained")
print(f"  Anomaly features : {len(anomaly_feats)}")
print(f"  Anomalies found  : {n_anom} ({n_anom/len(df)*100:.1f}% of district-months)")
print()
print("Top 10 most anomalous district-months:")
print(
    df[df["is_anomaly"]==1][["district","region","year","month","anomaly_score","ipc_phase"]]
    .sort_values("anomaly_score", ascending=False).head(10).to_string(index=False)
)

joblib.dump(anomaly_model, MDL / "anomaly_detector_if.pkl")
joblib.dump(scaler,        MDL / "anomaly_scaler.pkl")
joblib.dump(anomaly_feats, MDL / "anomaly_feature_cols.pkl")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

thresh = df[df["is_anomaly"]==1]["anomaly_score"].min()
axes[0].hist(df["anomaly_score"], bins=60, color=BLUE, alpha=0.85, edgecolor="white")
axes[0].axvline(thresh, color="red", lw=2, ls="--", label=f"Threshold ({thresh:.3f})")
axes[0].set_xlabel("Anomaly Score"); axes[0].set_ylabel("Count")
axes[0].set_title("Isolation Forest — Score Distribution"); axes[0].legend()

anom_r = df[df["is_anomaly"]==1].groupby("region")["is_anomaly"].count().sort_values()
axes[1].barh(anom_r.index, anom_r.values, color=BLUE, alpha=0.85)
axes[1].set_xlabel("Anomalous District-Months")
axes[1].set_title("Anomalies by Region")

plt.suptitle("FloodHunger Ghana — Anomaly Detection", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT / "model3_anomaly_results.png", bbox_inches="tight")
plt.show()

## 7 · SHAP Explainability

In [ ]:
print("Computing SHAP values...")
explainer   = shap.TreeExplainer(ipc_model)
shap_sample = X_test.sample(min(400, len(X_test)), random_state=SEED)
shap_values = explainer.shap_values(shap_sample)

# Handle both binary (2D array) and multiclass (3D array or list of 2D arrays)
if isinstance(shap_values, list):
    # List of 2D arrays — one per class; take mean absolute across all classes
    sv = np.mean([np.abs(sv_c) for sv_c in shap_values], axis=0)
elif shap_values.ndim == 3:
    # 3D array (n_samples, n_features, n_classes) — mean across classes
    sv = np.abs(shap_values).mean(axis=2)
else:
    sv = np.abs(shap_values)

feat_imp  = sv.mean(axis=0)
top_n     = 15
top_idx   = np.argsort(feat_imp)[-top_n:][::-1]
top_feats = [shap_sample.columns[i] for i in top_idx]
top_vals  = feat_imp[top_idx]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(range(top_n), top_vals[::-1], color=BLUE, alpha=0.85)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_feats[::-1], fontsize=10)
ax.set_xlabel("Mean |SHAP Value|  — Average impact on IPC phase prediction", fontsize=11)
ax.set_title("FloodHunger Ghana — Top 15 Features by SHAP\n(IPC Phase Classifier)", fontsize=13)
for bar, val in zip(bars, top_vals[::-1]):
    ax.text(bar.get_width() + 0.0003, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(OUT / "model1_shap_importance.png", bbox_inches="tight")
plt.show()
print(f"✓ Top 3 features: {top_feats[:3]}")

## 8 · Predictions & WFP Alert Generation

In [ ]:
df["pred_ipc_phase"]      = [idx_to_phase[i] for i in ipc_model.predict(X)]
df["pred_ipc_label"]      = df["pred_ipc_phase"].map(IPC_LABELS)
df["pred_ipc_action"]     = df["pred_ipc_phase"].map(IPC_ACTIONS)
df["pred_ipc_confidence"] = ipc_model.predict_proba(X).max(axis=1).round(3)
df["pred_price_change"]   = price_model.predict(df[reg_features].fillna(0)).round(2)
df["pred_price_direction"] = df["pred_price_change"].apply(
    lambda x: "↑ Rising" if x > 5 else ("↓ Falling" if x < -5 else "→ Stable")
)

df.to_csv(OUT / "predictions_full.csv", index=False)
print(f"✓ Full predictions saved: {len(df):,} rows")
print()
print("Predicted IPC Phase Distribution:")
for ph, cnt in df["pred_ipc_phase"].value_counts().sort_index().items():
    pct = cnt / len(df) * 100
    bar = "█" * int(pct / 2)
    print(f"  {IPC_COLORS[ph]} Phase {ph} — {IPC_LABELS[ph]:<12}: {cnt:>6,} ({pct:4.1f}%)  {bar}")

In [ ]:
max_year  = int(df["year"].max())
max_month = int(df[df["year"]==max_year]["month"].max())
month_str = datetime(max_year, max_month, 1).strftime("%B %Y")

latest = df[(df["year"]==max_year) & (df["month"]==max_month)].copy()
alerts = latest[[
    "district","region","pred_ipc_phase","pred_ipc_label",
    "pred_ipc_confidence","pred_ipc_action","pred_price_change",
    "pred_price_direction","is_anomaly","anomaly_score",
    "rainfall_anomaly_pct","flood_flag","compound_risk_score"
]].sort_values(["pred_ipc_phase","is_anomaly"], ascending=[False,False])

alerts.to_csv(OUT / "alerts_wfp_latest.csv", index=False)

print(f"WFP DISTRICT ALERT SNAPSHOT — {month_str}")
print("=" * 75)
print(f"{'District':<32} {'Region':<15} {'Phase':<8} {'Conf':<7} {'Price':<14} {'Anomaly'}")
print("-" * 75)
for _, r in alerts.iterrows():
    ph   = int(r["pred_ipc_phase"])
    icon = IPC_COLORS.get(ph,"⚪")
    anom = "⚠️ ANOMALY" if r["is_anomaly"]==1 else ""
    print(f"{icon} {r['district']:<30} {r['region']:<15} P{ph}       {r['pred_ipc_confidence']:.3f}  {r['pred_price_direction']:<14} {anom}")

In [ ]:
print()
print("=" * 65)
print("  FARMER VOICE ALERT BULLETIN")
print(f"  {month_str}  |  Broadcast via Africa's Talking SMS gateway")
print("=" * 65)

high_risk = alerts[alerts["pred_ipc_phase"] >= 3]
anomalous = alerts[(alerts["pred_ipc_phase"] < 3) & (alerts["is_anomaly"]==1)]

if len(high_risk) == 0 and len(anomalous) == 0:
    print("  ✅ No high-risk districts this month.")
else:
    if len(high_risk) > 0:
        print(f"\n  🔴 {len(high_risk)} DISTRICTS IN CRISIS OR EMERGENCY")
        for _, r in high_risk.iterrows():
            rain     = r["rainfall_anomaly_pct"]
            weather  = ("Heavy flooding detected." if r["flood_flag"]==1 else
                        "Severe dry spell." if rain < -30 else "Unstable weather conditions.")
            price_msg = ("Food prices are rising — store food now or buy in advance."
                         if "Rising" in str(r["pred_price_direction"]) else
                         "Food prices are currently stable.")
            print(f"\n  {IPC_COLORS[int(r['pred_ipc_phase'])]} {r['district']} ({r['region']})")
            msg = f"{weather} {price_msg} Contact your district food officer."
            print('     📢 "' + msg + '"')

    if len(anomalous) > 0:
        print(f"\n  ⚠️  {len(anomalous)} DISTRICTS WITH UNUSUAL RISK PATTERNS — MONITOR CLOSELY")
        for _, r in anomalous.iterrows():
            print(f"     • {r['district']} ({r['region']})")

print()
print("  [Translate to Twi / Dagbani / Ewe / Hausa before broadcast]")
print("=" * 65)

## 9 · Final Summary

In [ ]:
print("=" * 65)
print("  FLOODHUNGER GHANA — PIPELINE COMPLETE")
print("=" * 65)
print(f"  Districts modelled : {df['district'].nunique()}")
print(f"  Regions covered    : {df['region'].nunique()}")
print(f"  Total rows         : {len(df):,}")
print(f"  Years              : {int(df['year'].min())} – {int(df['year'].max())}")
print(f"  Features built     : {len(feature_cols)}")
print()
print("  MODEL RESULTS")
print(f"  IPC Classifier (XGBoost)       Weighted F1 : {f1:.4f}")
print(f"  Price Regressor (RandomForest) MAE         : {mae:.3f}%   R² : {r2:.4f}")
print(f"  Anomaly Detector (IsoForest)   Anomalies   : {n_anom} district-months")
print()
print("  OUTPUT FILES")
outputs = [
    ("data/outputs/predictions_full.csv",      "All district-month predictions"),
    ("data/outputs/alerts_wfp_latest.csv",     "WFP alert feed — latest month"),
    ("data/outputs/00_raw_data_overview.png",  "Raw data overview"),
    ("data/outputs/01_eda_overview.png",        "EDA (6 panels)"),
    ("data/outputs/02_seasonal_patterns.png",  "Seasonal IPC & flood patterns"),
    ("data/outputs/model1_confusion_matrix.png","IPC confusion matrix"),
    ("data/outputs/model1_shap_importance.png","SHAP feature importance"),
    ("data/outputs/model2_price_regressor.png","Price regressor evaluation"),
    ("data/outputs/model3_anomaly_results.png","Anomaly detection results"),
    ("data/models/ipc_classifier_xgb.pkl",    "Trained IPC model"),
    ("data/models/price_regressor_rf.pkl",     "Trained price model"),
    ("data/models/anomaly_detector_if.pkl",    "Trained anomaly model"),
]
for path, desc in outputs:
    print(f"  ✓ {path:<45} {desc}")
print("=" * 65)